# 5. Preprocesamiento

La variable objetivo es el **número diario de víctimas de hurto a personas registradas en Bogotá**, y sale del **registro anual de la Policía Nacional** (una fila por víctima, `CANTIDAD = 1`), no del agregado diario de MinDefensa. Lo fija la decisión D-01 en `docs/decisiones.md`.

Este notebook **ya no reconstruye la serie**: la lee de `data/processed/`, que produce `scripts/preprocesamiento.py`. Así el pipeline es reproducible fuera del notebook y los conteos de cada paso quedan registrados en `outputs/preprocesamiento/conteos_por_paso.csv`.

Para generarla:

```sh
python scripts/policia_registro_a_csv.py     # convierte los xlsx a CSV en streaming
python scripts/preprocesamiento.py           # construye los conjuntos procesados
```

La serie va del **2020-01-01 al 2025-10-31**, son 2.131 días. No llega a diciembre de 2025 porque noviembre y diciembre están incompletos en el archivo de la Policía y la caída que muestran es un artefacto: `docs/verificacion_caida_2025.md` (issue #86). El agregado de MinDefensa se conserva como control de consistencia, no como fuente de la serie.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# En Colab, sitúe el directorio de trabajo en la raíz del repositorio.
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
raw = root / 'data' / 'raw'
processed = root / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

def input_path(name):
    for folder in [root / 'DataSet', raw]:
        path = folder / name
        if path.is_file():
            return path
    return raw / name

In [ ]:
import json
import numpy as np

# La serie la produce scripts/preprocesamiento.py, que deja el rastro de conteos por paso.
path = processed / 'serie_objetivo_diaria.csv'
if not path.is_file():
    raise FileNotFoundError(
        f'Falta {path}. Ejecute primero:\n'
        f'  python scripts/policia_registro_a_csv.py\n'
        f'  python scripts/preprocesamiento.py'
    )

daily = pd.read_csv(path, parse_dates=['fecha']).set_index('fecha').sort_index()
print('Archivo:', path, '| Días:', len(daily), '| Víctimas:', f"{int(daily['victimas'].sum()):,}")

# Comprobaciones de integridad de la serie, no del archivo crudo.
calendario = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
faltantes = calendario.difference(daily.index)
print('Fechas ausentes del calendario:', len(faltantes))
print('Fechas repetidas:', int(daily.index.duplicated().sum()))
print('Días con cero víctimas:', int((daily['victimas'] == 0).sum()))
display(pd.DataFrame({'tipo': daily.dtypes.astype(str), 'nulos': daily.isna().sum()}))

# Evidencia del pipeline: de dónde salen estas filas.
resumen = root / 'outputs' / 'preprocesamiento' / 'resumen.json'
if resumen.is_file():
    meta = json.loads(resumen.read_text(encoding='utf-8'))['serie_objetivo']
    print('Periodo:', meta['periodo'], '| Filas de hurto antes del corte de periodo:',
          f"{meta['filas_hurto']:,}")
    display(pd.Series(meta['total_por_anio'], name='víctimas por año'))

# El agregado de MinDefensa se usa solo como control de consistencia.
control = input_path('hurto_bogota.csv')
if control.is_file():
    agg = pd.read_csv(control, dtype={'cod_muni': 'string'})
    agg['fecha'] = pd.to_datetime(agg['fecha_hecho'], format='ISO8601', errors='coerce').dt.normalize()
    comparacion = (agg.groupby('fecha')['cantidad'].sum()
                   .reindex(daily.index).rename('agregado').to_frame()
                   .join(daily['victimas'].rename('registro')))
    difieren = comparacion[comparacion['agregado'] != comparacion['registro']]
    print('Días en que el agregado y el registro difieren:', len(difieren), 'de', len(comparacion))
    if len(difieren):
        display(difieren.head())

Se exige una observación válida por fecha para que un desplazamiento de una fila equivalga a un día. Los huecos y duplicados se reportan para resolución con la fuente. Los picos se marcan con el percentil 99 de 2018–2024 como criterio descriptivo y se conservan. La marca queda fuera de los predictores.

In [ ]:
# Marca descriptiva de picos. El umbral usa 2020-2024 para no mirar el tramo de evaluación.
threshold = daily.loc[:'2024-12-31', 'victimas'].quantile(.99)
daily['pico_revision'] = daily['victimas'].gt(threshold)
print('Umbral p99 (2020-2024):', threshold, '| Días marcados:', int(daily['pico_revision'].sum()))
display(daily.nlargest(5, 'victimas'))

# Atributos de calendario y rezagos. El detalle y la prevención de fuga van en el issue #33.
daily['mes'] = daily.index.month
daily['dia_semana'] = daily.index.dayofweek
daily['fin_semana'] = daily['dia_semana'].ge(5)
for lag in [1, 7, 14]:
    daily[f'rezago_{lag}'] = daily['victimas'].shift(lag)
daily['media_previa_7'] = daily['victimas'].shift(1).rolling(7).mean()
display(daily.isna().sum().rename('nulos_por_columna'))
daily.to_csv(processed / 'hurto_diario.csv')
print('Escrito:', processed / 'hurto_diario.csv', '| columna objetivo: victimas')

Cada fila tiene como objetivo la cantidad del día indicado. Los primeros 1, 7 y 14 días carecen del histórico necesario para el rezago correspondiente. La media de los siete días anteriores también queda vacía al inicio durante siete días. Estos vacíos son esperados y se conservan. El conjunto común para evaluar los tres comparadores exige los 14 días de historia.